In [ ]:
import sys
import torch
from time import time
# Import custom utilities
sys.path.append('../utilities/')
# Import custom utilities
from utils_baseline import BaseUtilsCola
import pandas as pd
from transformers import set_seed

In [ ]:
set_seed(42)
torch.manual_seed(42)
batch_size = 32
epochs = 4
lr=5e-5
shuffle = True
clean_text = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
selection_criterion = 'eval_loss' # Choose between 'eval_matthews_correlation' and 'eval_loss'

In [ ]:
base_utils_obj = BaseUtilsCola()

In [ ]:
tokenized_train,tokenized_val,tokenized_test = base_utils_obj.get_tokenized_datasets()

In [ ]:
results_path = './results/batch_size_{}_epochs_{}_lr_{}_selection_criterion_{}'.format(batch_size, epochs, lr, selection_criterion)
log_path = './logs/batch_size_{}_epochs_{}_lr_{}_selection_criterion_{}'.format(batch_size, epochs, lr, selection_criterion)

In [ ]:
from transformers import TrainingArguments, Trainer,DistilBertForSequenceClassification

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2,device_map=device)

In [ ]:
# First, freeze everything
for param in model.distilbert.parameters():
    param.requires_grad = False

# Then, unfreeze *only* bias parameters in DistilBert
for name, param in model.distilbert.named_parameters():
    if 'bias' in name:
        param.requires_grad = True

# Finally, make sure the final classifier is trainable
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
training_args=TrainingArguments(
    output_dir=results_path,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=256,
    learning_rate=lr,
    logging_dir=log_path,
    logging_steps=10,
    eval_strategy='steps',
    save_steps=10,
    eval_steps=10,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model=selection_criterion,
    greater_is_better=False,
    report_to='tensorboard',
    seed=42,
    run_name='Batch size: {}, Epochs: {}, LR: {}, selection_criterion: {}'.format(batch_size, epochs, lr, selection_criterion),
    lr_scheduler_type='constant',
    warmup_steps=0,
    fp16 = False
)

In [ ]:
start_time = time()

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=base_utils_obj.tokenizer,
    compute_metrics=base_utils_obj.compute_metric_mcc
)

In [ ]:
trainer.train()

In [ ]:
# Save the best model
'''
Keep a local copy of the best model
'''
best_model_path = "./best_model_{batch_size}_{epochs}_{lr}_selection_criterion_{selection_criterion}".format(batch_size=batch_size, epochs=epochs, lr=lr, selection_criterion=selection_criterion)
trainer.model.save_pretrained(best_model_path)

In [ ]:
base_utils_obj.write_time(start_time,time(),batch_size,epochs)

In [ ]:
df1 = pd.read_csv("./cola_in_domain_test.tsv", sep="\t")
df2 = pd.read_csv("./cola_out_of_domain_test.tsv", sep="\t")
df1 = df1.rename(columns={"Sentence": "sentence"})
df2 = df2.rename(columns={"Sentence": "sentence"})

In [ ]:
from datasets import Dataset
test_in_domain = Dataset.from_pandas(df1[['sentence']])
test_out_of_domain = Dataset.from_pandas(df2[['sentence']])
test_in_domain = test_in_domain.map(base_utils_obj.tokenize_function, batched=True)
test_out_of_domain = test_out_of_domain.map(base_utils_obj.tokenize_function, batched=True)

In [ ]:
in_domain_predictions = trainer.predict(test_in_domain)
out_of_domain_predictions = trainer.predict(test_out_of_domain)
in_domain_pred_labels = in_domain_predictions.predictions.argmax(-1)
out_of_domain_pred_labels = out_of_domain_predictions.predictions.argmax(-1)

In [ ]:
base_utils_obj.save_predictions(in_domain_pred_labels,name='in_domain_predictions')
base_utils_obj.save_predictions(out_of_domain_pred_labels,name='out_of_domain_predictions')